In [ ]:
#将12个聚类的预测结果合并到一个文件中（12个csv---＞1个中的csv）
import os
import pandas as pd

# 指定输入文件夹路径和输出文件夹路径
input_folder_path = 'D:\DESKTOP\desk\\bianliang\esm-ssp585\p12'
output_file_path = 'D:\DESKTOP\desk\\bianliang\esm-ssp585\p12\p12ALL.csv'

# 创建一个空的 DataFrame 用于存储合并的结果
combined_df = pd.DataFrame()

# 遍历输入文件夹中的所有文件
for filename in os.listdir(input_folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder_path, filename)
        
        # 读取 CSV 文件
        df = pd.read_csv(file_path)
        
        # 合并到总 DataFrame 中
        combined_df = pd.concat([combined_df, df], ignore_index=True)

# 保存结果到新的 CSV 文件
combined_df.to_csv(output_file_path, index=False)

print(f"已成功复制所有 CSV 文件的内容，结果保存在 '{output_file_path}'。")


In [ ]:
#计算全球每年的年平均水文和径流变量趋势（用于画折线图）；P、ET、R、n
import pandas as pd
import os

# 输入文件路径
input_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\p12\\p12ALL.csv'

# 输出文件路径
output_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\R-trend\\p12ALL-trend-585.csv'

# 确保输出文件夹存在
output_dir = os.path.dirname(output_file)
os.makedirs(output_dir, exist_ok=True)

# 读取CSV文件
df = pd.read_csv(input_file)

# 需要加权的变量
variables = ['PPT', 'PET', 'R', 'n', 'AET']

# 创建一个空的DataFrame来存放加权结果
weighted_results = pd.DataFrame()

# 按年份分组并计算每个变量的面积加权平均
weighted_results['year'] = df['year'].unique()
for var in variables:
    weighted_var = df.groupby('year').apply(lambda x: (x[var] * x['AREA']).sum() / x['AREA'].sum()).reset_index(name=var)
    weighted_results = pd.merge(weighted_results, weighted_var, on='year')

# 保存到新的CSV文件
weighted_results.to_csv(output_file, index=False)

print("加权结果已保存到：", output_file)


In [ ]:
#全球平均归因趋势（用于画折线图）；▲R、▲Rsub<sub>P</sub>、▲Rsub<sub>PET</sub>、▲Rsub<sub>n</sub>
import pandas as pd
import numpy as np
import math

# 定义 Choudhury-Yang Budyko 模型函数
def budyko(PPT, PET, n):
    phi = PET / PPT
    ET = PPT * (1 + phi - (1 + phi**n)**(1/n))
    R = PPT - ET
    return R

# 定义弹性系数计算公式
def compute_elasticities(PPT, PET,  n):
    phi = PET / PPT
    # 公式推导的弹性系数
    eps_ppt =((1+phi**n)**(1/n+1)-phi**(n+1))/((1+phi**n)*((1+phi**n)**(1/n)-phi))
    eps_pet =1/((1+phi**n)*((1-(1+phi**-n)**(1/n))))
    eps_n =(math.log(1+phi**n)+phi**n*math.log(1+phi**-n))/(n*(1+phi**n)*(1-(1+phi**-n)**(1/n)))
    return eps_ppt, eps_pet, eps_n

# 读取数据
input_file = 'D:\DESKTOP\desk\\bianliang\esm-ssp585-ssp126Lu\\attribution\guiyin\p12ALL-trend-585.csv'
output_file = 'D:\DESKTOP\desk\\bianliang\esm-ssp585-ssp126Lu\\attribution\guiyin\esm-ssp585-out.csv'

df = pd.read_csv(input_file)

# 全时域 2015-2100
all_period = df[(df['year'] >= 2015) & (df['year'] <= 2100)]

# 全时域取平均
PPT00 = all_period['PPT'].mean()
PET00 = all_period['PET'].mean()
n00 = all_period['n'].mean()
R00 = all_period['R'].mean()

# 计算全时域弹性系数
eps_ppt00, eps_pet00, eps_n00 = compute_elasticities(PPT00, PET00, n00)

# 筛选基准期 1985-2014
base_period = df[(df['year'] >= 1985) & (df['year'] <= 2014)]

# 基准期取平均
PPT0 = base_period['PPT'].mean()
PET0 = base_period['PET'].mean()
n0 = base_period['n'].mean()
R0 = base_period['R'].mean()

# 计算基准期弹性系数
eps_ppt0, eps_pet0, eps_n0 = compute_elasticities(PPT0, PET0, n0)

# 初始化结果列表
results = []

# 变化期，从1986年开始每30年滑动
start_year = 1986
end_year = df['year'].max()

while start_year + 30 - 1 <= end_year:
    period = df[(df['year'] >= start_year) & (df['year'] <= start_year + 30 - 1)]
    
    PPT1 = period['PPT'].mean()
    PET1 = period['PET'].mean()
    n1 = period['n'].mean()
    R1 = period['R'].mean()

    # 变化期弹性系数
    eps_ppt1, eps_pet1, eps_n1 = compute_elasticities(PPT1, PET1, n1)

    # 使用基准期弹性系数进行归因
    delta_PPT = PPT1 - PPT0
    delta_PET = PET1 - PET0
    delta_n = n1 - n0

    delta_RPPT = delta_PPT / PPT00 * eps_ppt00 * R00
    delta_RPET = delta_PET / PET00 * eps_pet00 * R00
    delta_Rn = delta_n / n00 * eps_n00 * R00
    delta_RCC = delta_RPPT + delta_RPET
    delta_R_total = delta_RPPT + delta_RPET + delta_Rn

    # 贡献率（百分比）
    if delta_R_total != 0:
        delta_PPT_percent = delta_RPPT / delta_R_total * 100
        delta_PET_percent = delta_RPET / delta_R_total * 100
        delta_n_percent = delta_Rn / delta_R_total * 100
        delta_CC_percent = delta_PPT_percent + delta_PET_percent
    else:
        delta_PPT_percent = delta_PET_percent = delta_n_percent = np.nan

    # 保存结果
    results.append([
        start_year + 30 - 1,  # 窗口最后一年
        #eps_ppt00, eps_pet00, eps_n00,    # 全时域弹性系数
        #eps_ppt0, eps_pet0, eps_n0,    # 基准期弹性系数
        eps_ppt1, eps_pet1, eps_n1,
        delta_RPPT, delta_RPET,delta_RCC, delta_Rn, delta_R_total,
        delta_PPT_percent, delta_PET_percent,delta_CC_percent, delta_n_percent
    ])

    # 往后滑动一年
    start_year += 1

# 转成 DataFrame 并保存，esp_xx：弹性；dRxx：各要素引起的径流变化量（▲Rxx）；pxx：各要素的变化量
columns = [
    'Year', 
    #'εPPT00', 'εPET00', 'εn00',
    #'εPPT0', 'εPET0', 'εn0',
    'eps_PPT', 'eps_PET', 'eps_n',
    'dRPPT', 'dRPET', 'dRCC', 'dRn', 'dR',
    'pPPT', 'pPET', 'pCC', 'pn'
]
output_df = pd.DataFrame(results, columns=columns)
output_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"计算完成！结果保存到 {output_file}")